In [8]:
import numpy as np
import glob as glob
import json

import matplotlib.pyplot as plt

In [2]:
def clean_dictionary(dd, keep_keys=None):
    ''' Takes in dictionary, outputs dictionary with only the keys in keep_keys'''
    out = {}
    for key in dd.keys():
        if key in keep_keys:
            out[key] = dd[key]
    return out

In [3]:
'''
fbase = '/home/puehlengt/appletree/notebooks/6_params'
prefix = 'testsims_6params'
#'''

fbase = '/home/puehlengt/appletree/notebooks/3_params_uniform_g1/training_set'
prefix = 'testsims_3params_uniformg1'

needs_cleaning = False
num_sims = 20000
fnames = glob.glob(f'{fbase}/{prefix}_{num_sims:.0f}sims*.npy')

# open one file to get structure
bag = np.load(fnames[0], allow_pickle=True).item()

if needs_cleaning:
    # find which parameters are floated
    floated_params = []
    for key, val in bag['param_bag'][1].items():
        if isinstance(val, np.float64): # if it's a floated parameter
            floated_params.append(key)

    bag['param_bag'] = [clean_dictionary(_, keep_keys=floated_params) for _ in bag['param_bag']]

In [4]:
for _f in fnames[1:]:
    tmp_dict = np.load(_f, allow_pickle=True).item()

    if needs_cleaning:
        tmp_dict['param_bag'] = [clean_dictionary(_, keep_keys=floated_params) for _ in tmp_dict['param_bag']]

    # appending 
    for k in bag.keys():
        bag[k].extend(tmp_dict[k])

In [5]:
assert len(bag['events_bag']) == num_sims*len(fnames), 'Missing some simulations.'

In [6]:
save_on = True
#save_on = False
save_name = f'{fbase}/harvested_{prefix}.npy'

print(f'Save path: {save_name}')
if save_on:
    print('Saving...')
    np.save(save_name, bag)
    print('Done.')

Save path: /home/puehlengt/appletree/notebooks/3_params_uniform_g1/training_set/harvested_testsims_3params_uniformg1.npy
Saving...
Done.


In [10]:
# same param config used when running the sims
f_param_config = '/home/puehlengt/appletree/notebooks/param_nr_sr1_3params_uniformg1.json'

with open(f_param_config) as fid:
    param_config = json.load(fid)

# floated params are exactly the keys that appear in param_bag
floated_params = list(bag['param_bag'][0].keys())

prior_config = {}
for par in floated_params:
    cfg = param_config[par]
    prior_type = cfg['prior_type']

    if prior_type == 'uniform':
        prior_args = cfg['prior_args']  # already has lower/upper
    elif prior_type == 'norm':
        prior_args = cfg['prior_args']  # already has mean/std
    elif prior_type == 'free':
        prior_args = {'mean': cfg['init_mean'], 'std': cfg['init_std']}
        prior_type = 'norm'

    prior_config[par] = {
        'prior_type':    prior_type,
        'prior_args':    prior_args,
        'allowed_range': cfg['allowed_range'],
    }

prior_save_path = f'{fbase}/prior_{prefix}.json'
print(f'Floated params: {floated_params}')
print(f'Prior config:\n{json.dumps(prior_config, indent=4)}')
print(f'Save path: {prior_save_path}')
if save_on:
    with open(prior_save_path, 'w') as fid:
        json.dump(prior_config, fid, indent=4)
    print('Saved.')

Floated params: ['g1', 'g2', 'ambe_nr_rate']
Prior config:
{
    "g1": {
        "prior_type": "uniform",
        "prior_args": {
            "lower": 0.12,
            "upper": 0.14
        },
        "allowed_range": [
            0,
            1.0
        ]
    },
    "g2": {
        "prior_type": "norm",
        "prior_args": {
            "mean": 16.85,
            "std": 0.46
        },
        "allowed_range": [
            0,
            100.0
        ]
    },
    "ambe_nr_rate": {
        "prior_type": "norm",
        "prior_args": {
            "mean": 5500,
            "std": 100
        },
        "allowed_range": [
            0,
            10000000000.0
        ]
    }
}
Save path: /home/puehlengt/appletree/notebooks/3_params_uniform_g1/training_set/prior_testsims_3params_uniformg1.json
Saved.


In [ ]:
raise

In [ ]:
n = [np.shape(tmp)[1] for tmp in bag['events_bag']]
rate = [tmp['ambe_nr_rate'] for tmp in bag['param_bag']]
plt.hist(rate, bins=100, histtype='step', density=True, label='rate')
plt.hist(n, bins=100, histtype='step', density=True, label='n')
plt.title(f'{len(bag["events_bag"]):.1e} pseudo-experiments')
plt.legend()